In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Customer Segmentation using K-Means Clustering

## Objective
Segment customers based on income and spending patterns to identify target groups for targeted marketing strategies.

## Dataset
**Mall Customer Segmentation Data** from Kaggle

### Features
- **Age**: Customer age
- **Annual Income**: Customer's annual income in thousands
- **Spending Score**: Score assigned based on customer behavior and spending nature (1-100)


## Section 1: Data Loading and Understanding

In [ ]:
# Download dataset from Kaggle if needed
# For first-time use, download from: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
# Place the 'Mall_Customers.csv' file in the 'data' folder

# Load the dataset
try:
    df = pd.read_csv('data/Mall_Customers.csv')
except FileNotFoundError:
    print("Dataset not found. Please download 'Mall_Customers.csv' from Kaggle:")
    print("https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python")
    print("\nPlace it in the 'data' folder and run this cell again.")
    df = None

if df is not None:
    # Clean column names - remove any backslashes and extra whitespace
    df.columns = df.columns.str.replace('\\', '', regex=False).str.strip()
    print("Dataset loaded successfully!")
    print(f"\nDataset Shape: {df.shape}")
    print(f"\nFirst few rows:")
    print(df.head())


In [3]:
# Check dataset information
if df is not None:
    print("\n=== Dataset Information ===")
    print(df.info())
    print("\n=== Missing Values ===")
    print(df.isnull().sum())
    print("\n=== Basic Statistics ===")
    print(df.describe())


=== Dataset Information ===
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              200 non-null    int64  
 1   Gender                  200 non-null    str    
 2   Age                     200 non-null    int64  
 3   Annual Income (k\$)     200 non-null    float64
 4   Spending Score (1-100)  200 non-null    float64
dtypes: float64(2), int64(2), str(1)
memory usage: 7.9 KB
None

=== Missing Values ===
CustomerID                0
Gender                    0
Age                       0
Annual Income (k\$)       0
Spending Score (1-100)    0
dtype: int64

=== Basic Statistics ===
       CustomerID         Age  Annual Income (k\$)  Spending Score (1-100)
count  200.000000  200.000000           200.000000              200.000000
mean   100.500000   39.880000            80.869611               46.258340
std     57.87918

In [ ]:
# Display column names
if df is not None:
    print("Column names in the dataset:")
    print(df.columns.tolist())
    print("\nDatatype of each column:")
    print(df.dtypes)

## Section 2: Exploratory Data Analysis (EDA)

In [ ]:
# Visualization 1: Age vs Spending Score
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Age vs Spending Score
    axes[0].scatter(df['Age'], df['Spending Score (1-100)'], alpha=0.6, s=50, color='steelblue')
    axes[0].set_xlabel('Age', fontsize=12)
    axes[0].set_ylabel('Spending Score', fontsize=12)
    axes[0].set_title('Age vs Spending Score', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Annual Income vs Spending Score
    axes[1].scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], alpha=0.6, s=50, color='coral')
    axes[1].set_xlabel('Annual Income (k$)', fontsize=12)
    axes[1].set_ylabel('Spending Score', fontsize=12)
    axes[1].set_title('Annual Income vs Spending Score', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Key Observations:")
    print("- Look for patterns of high spenders (high scores) vs low spenders")
    print("- Note the relationship between income and spending behavior")

In [ ]:
# Visualization 2: Distribution plots
if df is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(df['Age'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Age')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Age Distribution')
    
    axes[1].hist(df['Annual Income (k$)'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Annual Income (k$)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Annual Income Distribution')
    
    axes[2].hist(df['Spending Score (1-100)'], bins=30, color='salmon', edgecolor='black', alpha=0.7)
    axes[2].set_xlabel('Spending Score')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Spending Score Distribution')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Statistical Summary by Income and Spending
if df is not None:
    print("=== High Spenders vs Low Spenders ===")
    high_spend_threshold = df['Spending Score (1-100)'].quantile(0.75)
    low_spend_threshold = df['Spending Score (1-100)'].quantile(0.25)
    
    high_spenders = df[df['Spending Score (1-100)'] >= high_spend_threshold]
    low_spenders = df[df['Spending Score (1-100)'] <= low_spend_threshold]
    
    print(f"\nHigh Spenders (Top 25%, Score >= {high_spend_threshold:.1f}):")
    print(f"  Count: {len(high_spenders)}")
    print(f"  Avg Age: {high_spenders['Age'].mean():.1f}")
    print(f"  Avg Income: ${high_spenders['Annual Income (k$)'].mean():.1f}k")
    print(f"  Avg Spending Score: {high_spenders['Spending Score (1-100)'].mean():.1f}")
    
    print(f"\nLow Spenders (Bottom 25%, Score <= {low_spend_threshold:.1f}):")
    print(f"  Count: {len(low_spenders)}")
    print(f"  Avg Age: {low_spenders['Age'].mean():.1f}")
    print(f"  Avg Income: ${low_spenders['Annual Income (k$)'].mean():.1f}k")
    print(f"  Avg Spending Score: {low_spenders['Spending Score (1-100)'].mean():.1f}")

## Section 3: Feature Selection and Preprocessing

In [ ]:
# Select features for clustering
if df is not None:
    # Select Annual Income and Spending Score for clustering
    X = df[['Annual Income (k$)', 'Spending Score (1-100)']].copy()
    
    print("Features selected for clustering:")
    print(f"1. Annual Income (k$)")
    print(f"2. Spending Score (1-100)")
    print(f"\nFeature matrix shape: {X.shape}")
    print(f"\nFeature statistics before scaling:")
    print(X.describe())


KeyError: "['Annual Income (k$)'] not in index"

In [5]:
# Feature Scaling using StandardScaler
if df is not None:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print("Feature Scaling Applied (StandardScaler)")
    print(f"\nScaled features shape: {X_scaled.shape}")
    print(f"\nMean of scaled features (should be ~0): {X_scaled.mean(axis=0)}")
    print(f"Std of scaled features (should be ~1): {X_scaled.std(axis=0)}")
    
    # Create a DataFrame for scaled features
    X_scaled_df = pd.DataFrame(X_scaled, columns=['Annual Income (scaled)', 'Spending Score (scaled)'])
    print(f"\nFirst few rows of scaled data:")
    print(X_scaled_df.head())

NameError: name 'X' is not defined

## Section 4: Elbow Method for Optimal K

In [ ]:
# Elbow Method: Calculate WCSS for different K values
if df is not None:
    wcss = []
    K_range = range(1, 11)
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
        kmeans.fit(X_scaled)
        wcss.append(kmeans.inertia_)
    
    print("Within-Cluster Sum of Squares (WCSS) for different K values:")
    for k, w in zip(K_range, wcss):
        print(f"K={k}: WCSS={w:.2f}")
    
    # Plot the Elbow Curve
    plt.figure(figsize=(10, 6))
    plt.plot(K_range, wcss, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Number of Clusters (K)', fontsize=12)
    plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12)
    plt.title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.xticks(K_range)
    
    # Annotate the elbow point (typically K=5)
    plt.axvline(x=5, color='red', linestyle='--', linewidth=2, label='Optimal K=5')
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
    
    print("\n✓ The elbow appears around K=5, suggesting 5 is the optimal number of clusters")

## Section 5: K-Means Clustering Implementation

In [ ]:
# Train K-Means with optimal K=5
if df is not None:
    optimal_k = 5
    kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
    clusters = kmeans_final.fit_predict(X_scaled)
    
    print(f"K-Means Clustering Model Trained")
    print(f"Optimal number of clusters: {optimal_k}")
    print(f"Total iterations: {kmeans_final.n_iter_}")
    print(f"Final inertia (WCSS): {kmeans_final.inertia_:.2f}")
    
    # Add cluster labels to the original dataframe
    df['Cluster'] = clusters
    
    print(f"\nCluster Distribution:")
    print(df['Cluster'].value_counts().sort_index())
    
    # Calculate percentage of customers in each cluster
    print(f"\nCluster Distribution (%):")
    print((df['Cluster'].value_counts(normalize=True).sort_index() * 100).round(2))

In [ ]:
# Cluster Centers in original scale
if df is not None:
    cluster_centers = scaler.inverse_transform(kmeans_final.cluster_centers_)
    cluster_centers_df = pd.DataFrame(
        cluster_centers,
        columns=['Annual Income (k$)', 'Spending Score (1-100)']
    )
    cluster_centers_df.index.name = 'Cluster'
    
    print("Cluster Centers (in original scale):")
    print(cluster_centers_df.round(2))

## Section 6: Cluster Visualization and Interpretation

In [ ]:
# Visualization: Clusters in 2D space
if df is not None:
    plt.figure(figsize=(12, 7))
    
    # Color palette
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
    
    # Plot each cluster
    for cluster in range(optimal_k):
        cluster_data = df[df['Cluster'] == cluster]
        plt.scatter(
            cluster_data['Annual Income (k$)'],
            cluster_data['Spending Score (1-100)'],
            c=colors[cluster],
            label=f'Cluster {cluster}',
            s=100,
            alpha=0.6,
            edgecolors='black',
            linewidth=0.5
        )
    
    # Plot cluster centers
    plt.scatter(
        cluster_centers[:, 0],
        cluster_centers[:, 1],
        c='black',
        marker='X',
        s=300,
        edgecolors='yellow',
        linewidth=2,
        label='Centroids'
    )
    
    plt.xlabel('Annual Income (k$)', fontsize=12)
    plt.ylabel('Spending Score (1-100)', fontsize=12)
    plt.title('Customer Segments - K-Means Clustering (K=5)', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Detailed Cluster Analysis
if df is not None:
    print("="*70)
    print("CLUSTER ANALYSIS AND BUSINESS INSIGHTS")
    print("="*70)
    
    for cluster in range(optimal_k):
        cluster_data = df[df['Cluster'] == cluster]
        
        avg_age = cluster_data['Age'].mean()
        avg_income = cluster_data['Annual Income (k$)'].mean()
        avg_spending = cluster_data['Spending Score (1-100)'].mean()
        count = len(cluster_data)
        
        print(f"\n📊 CLUSTER {cluster}:")
        print(f"   Size: {count} customers ({count/len(df)*100:.1f}%)")
        print(f"   Average Age: {avg_age:.1f} years")
        print(f"   Average Income: ${avg_income:.1f}k")
        print(f"   Average Spending Score: {avg_spending:.1f}/100")
        
        # Classify cluster
        if avg_income > df['Annual Income (k$)'].median() and avg_spending > df['Spending Score (1-100)'].median():
            segment = "🎯 HIGH INCOME, HIGH SPENDERS (Premium Segment)"
        elif avg_income > df['Annual Income (k$)'].median() and avg_spending <= df['Spending Score (1-100)'].median():
            segment = "💼 HIGH INCOME, LOW SPENDERS (Potential Segment)"
        elif avg_income <= df['Annual Income (k$)'].median() and avg_spending > df['Spending Score (1-100)'].median():
            segment = "🛍️ LOW INCOME, HIGH SPENDERS (Target Segment)"
        else:
            segment = "📉 LOW INCOME, LOW SPENDERS (Budget Segment)"
        
        print(f"   Segment: {segment}")

In [ ]:
# Box plots for cluster comparison
if df is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Age by Cluster
    df.boxplot(column='Age', by='Cluster', ax=axes[0])
    axes[0].set_title('Age Distribution by Cluster')
    axes[0].set_xlabel('Cluster')
    axes[0].set_ylabel('Age')
    
    # Income by Cluster
    df.boxplot(column='Annual Income (k$)', by='Cluster', ax=axes[1])
    axes[1].set_title('Income Distribution by Cluster')
    axes[1].set_xlabel('Cluster')
    axes[1].set_ylabel('Annual Income (k$)')
    
    # Spending Score by Cluster
    df.boxplot(column='Spending Score (1-100)', by='Cluster', ax=axes[2])
    axes[2].set_title('Spending Score Distribution by Cluster')
    axes[2].set_xlabel('Cluster')
    axes[2].set_ylabel('Spending Score')
    
    plt.suptitle('')  # Remove automatic title
    plt.tight_layout()
    plt.show()

In [ ]:
# Summary Statistics Table
if df is not None:
    summary_table = pd.DataFrame()
    
    for cluster in range(optimal_k):
        cluster_data = df[df['Cluster'] == cluster]
        summary_table[f'Cluster {cluster}'] = [
            len(cluster_data),
            f"{len(cluster_data)/len(df)*100:.1f}%",
            f"{cluster_data['Age'].mean():.1f}",
            f"{cluster_data['Annual Income (k$)'].mean():.1f}",
            f"{cluster_data['Spending Score (1-100)'].mean():.1f}"
        ]
    
    summary_table.index = ['Count', 'Percentage', 'Avg Age', 'Avg Income (k$)', 'Avg Spending Score']
    
    print("\n" + "="*70)
    print("SUMMARY STATISTICS TABLE")
    print("="*70)
    print(summary_table)